In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('C:\\Users\\Admin\\Desktop\\Data_pandas/Advertising.csv',index_col=0)

In [3]:
df.head()

,TV,radio,newspaper,sales
1,230.1,37.8,69.2,22.1
2,44.5,39.3,45.1,10.4
3,17.2,45.9,69.3,9.3
4,151.5,41.3,58.5,18.5
5,180.8,10.8,58.4,12.9


In [4]:
X = df['TV']
y = df['sales']
n = len(y)
X = np.append(np.ones((n,1)), X.values.reshape(n,1), axis = 1)
y = df['sales'].values.reshape(n,1)
par = np.zeros((2,1))

In [5]:
def cost_function(X, y , par):
    y_pred = np.dot(X, par)
    error = (y_pred - y)**2
    cost = 1/(n)*np.sum(error)
    return cost

In [6]:
def grad_d(X,y, par, alpha, iterations):
    costs = []
    for i in range(iterations):
        y_pred = np.dot(X, par)
        der = np.dot (X.transpose(), (y_pred - y))/ n
        par -= alpha * der
        costs.append(cost_function(X,y, par))
    return par, costs

In [7]:
par, costs = grad_d(X,y, par, 0.00005, 500000)

In [8]:
par

array([[7.02008789],
       [0.04760015]])

In [9]:
import numpy as np
import statsmodels.api as sm
mod = sm.OLS(y, X)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.612
Model:                            OLS   Adj. R-squared:                  0.610
Method:                 Least Squares   F-statistic:                     312.1
Date:                Sat, 24 Jan 2026   Prob (F-statistic):           1.47e-42
Time:                        13:31:13   Log-Likelihood:                -519.05
No. Observations:                 200   AIC:                             1042.
Df Residuals:                     198   BIC:                             1049.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          7.0326      0.458     15.360      0.0

In [10]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X,y)
print(model.coef_)
print(model.intercept_)

[[0.         0.04753664]]
[7.03259355]


In [11]:
import pandas as pd
import numpy as np


X = df[['TV','radio','newspaper']]
y = df['sales']
n = len(y)
X = np.append(np.ones((n,1)), X.values.reshape(n,3), axis = 1)
y = df['sales'].values.reshape(n,1)
par = np.zeros((4,1))
print(par)
def cost_function(X, y , par):
    y_pred = np.dot(X, par)
    error = (y_pred - y)**2
    cost = 1/(n)*np.sum(error)
    return cost

def grad_d(X,y, par, alpha, iterations, eps=0.001):
    costs = []
    for i in range(iterations):
        y_pred = np.dot(X, par)
        der = np.dot (X.transpose(), (y_pred - y))/ n
        par -= alpha * der
        costs.append(cost_function(X,y, par))
        if np.linalg.norm(der) <= eps:
            break
    return par, costs

par, costs = grad_d(X,y, par, 0.00005, 500000)
print(par.round(3))

[[0.]
 [0.]
 [0.]
 [0.]]
[[ 2.863e+00]
 [ 4.600e-02]
 [ 1.890e-01]
 [-1.000e-03]]


градиентный спуск

In [12]:
import seaborn as sns
df = sns.load_dataset('diamonds')
df

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset('diamonds')

df.drop(['depth', 'table', 'x', 'y', 'z'], axis=1, inplace=True)
df = pd.get_dummies(df, drop_first=True)

df['carat'] = np.log(1+df['carat'])
df['price'] = np.log(1+df['price'])

X_cols = [col for col in df.columns if col!='price']
X = df[X_cols]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

parameters = {
    "loss": ["squared_error", "epsilon_insensitive"],
    "penalty": ["elasticnet"],
    "alpha": np.logspace(-3, 3, 10),
    "l1_ratio": np.linspace(0, 1, 10),
    "learning_rate": ["constant"],
    "eta0": np.logspace(-4, -1, 4)
}

sgd = SGDRegressor(random_state=42)
sgd_cv = GridSearchCV(estimator=sgd, param_grid=parameters, n_jobs=-1)
sgd_cv.fit(X_train, y_train)

print(sgd_cv.best_params_)

sgd = SGDRegressor(**sgd_cv.best_params_, random_state = 42)

sgd.fit(X_train, y_train)
sgd.score(X_train, y_train) # r2
ls = sgd.predict(X_test)

round(mean_squared_error(y_test, ls), 3)

In [13]:
def func1(x):
    return 6*x**5-5*x**4-4*x**3+3*x**2
 
def func2(x):
    return 30*x**4-20*x**3-12*x**2+6*x
init_value = 0.7
iter_count = 0
x_curr = init_value
epsilon = 0.000001
f = func1(x_curr)
 
while (abs(f) > epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(x_curr)
print(iter_count)

0.6296335078534031
0.6286680781673306
0.6286669787778999
0.6286669787764609
4


метод ньютона

In [14]:
def func1(x):
    return 3*x**2 - 6*x -45
def func2(x):
    return 6*x - 6
init_value = 42
iter_count = 0
x_curr = init_value
epsilon = 0.0001
f = func1(x_curr)
 
while (abs(f) > epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(x_curr)
print(iter_count)

21.695121951219512
11.734125501243229
7.1123493600499685
5.365000391507974
5.015260627016227
5.000029000201801
5.000000000105126
5.000000000000001
8


In [15]:
def func1(x):
    return x**3 - 72*x -220
def func2(x):
    return 3*x**2 - 72
def newtons_method(f, fprime, x0, tol=0.0001):
    iter_count = 0
    x_curr = x0
    f_val = f(x_curr)
    while (abs(f_val) > tol):
        f_val = f(x_curr)
        f_prime_val = fprime(x_curr)
        x_curr = x_curr - (f_val)/(f_prime_val)
        iter_count += 1
    return x_curr

newtons_method(f=func1, fprime=func2, x0=12, tol=0.0001)

9.727134419408875

Найдите положительный корень для уравнения 

In [16]:
def func1(x):
    return x**2 - 9*x -5
def func2(x):
    return 2*x - 9
def newtons_method(f, fprime, x0, tol=0.0001):
    iter_count = 0
    x_curr = x0
    f_val = f(x_curr)
    while (abs(f_val) > tol):
        f_val = f(x_curr)
        f_prime_val = fprime(x_curr)
        x_curr = x_curr - (f_val)/(f_prime_val)
        iter_count += 1
    return x_curr

newtons_method(f=func1, fprime=func2, x0=2.2, tol=0.0001)

-0.524937810560627

In [17]:
from scipy.optimize import newton
newton(func=func1, fprime=func2, x0=50, tol=0.0001)

np.float64(9.524937810565161)

In [18]:
def func(x):
    return 8*x**3-2*x**2-450
def func1(x):
    return 24*x**2 - 4*x 
def func2(x):
    return 48*x -4
def newtons_method(f, fprime, x0, tol=0.0001):
    iter_count = 0
    x_curr = x0
    f_val = f(x_curr)
    while (abs(f_val) > tol):
        f_val = f(x_curr)
        f_prime_val = fprime(x_curr)
        x_curr = x_curr - (f_val)/(f_prime_val)
        iter_count += 1
    return x_curr

newtons_method(f=func1, fprime=func2, x0=42, tol=0.0001)

0.16666666666666785

квазиньютоновские методы

In [19]:
import numpy as np
from scipy.optimize import minimize

In [20]:
def func(x):
    return x[0]**2.0 + x[1]**2.0
def grad_func(x):
    return np.array([x[0] * 2, x[1] * 2])
x_0 = [1.0, 1.0]
result = minimize(func, x_0, method='BFGS', jac=grad_func)
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))
 
#Статус оптимизации Optimization terminated successfully.
#Количество оценок: 3
#Решение: f([0. 0.]) = 0.00000

Статус оптимизации Optimization terminated successfully.
Количество оценок: 3
Решение: f([0. 0.]) = 0.00000


In [21]:
# определяем нашу функцию
def func(x):
    return x[0]**2.0 + x[1]**2.0
 
#  определяем градиент функции
def grad_func(x):
    return np.array([x[0] * 2, x[1] * 2])
 
# определяем начальную точку
x_0 = [1, 1]
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
Количество оценок: 3
Решение: f([0. 0.]) = 0.00000


In [22]:
# определяем нашу функцию
def func(x):
    return x[0]**2 - x[0]*x[1] + x[1]**2+9*x[0] -6*x[1] + 20
 
#  определяем градиент функции
def grad_func(x):
    return np.array([2 * x[0] - x[1] + 9, -x[0] + 2 * x[1] - 6])
 
# определяем начальную точку
x_0 = [-400, -400]
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
Количество оценок: 9
Решение: f([-3.99999972  1.00000028]) = -1.00000


In [23]:
def func(x):
    return x[0]**2 -3 *x[0] +45
def grad_func(x):
    return 2 * x[0] -3
x_0 = 10
result = minimize(func, x_0, method='BFGS', jac=grad_func)
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))
 
#Статус оптимизации Optimization terminated successfully.
#Количество оценок: 3
#Решение: f([0. 0.]) = 0.00000

Статус оптимизации Optimization terminated successfully.
Количество оценок: 5
Решение: f([1.5]) = 42.75000


In [24]:

from scipy.optimize import minimize

def func(x):
    return x[0]**2.0 - 3*x[0] + 45

def grad_func(x):
    return 2*x[0]-3

x_0 = 10
result = minimize(func, x_0, method='BFGS', jac=grad_func)
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации Optimization terminated successfully.
Количество оценок: 5
Решение: f([1.5]) = 42.75000


In [25]:

from scipy.optimize import minimize

def func(x):
    return x[0]**2.0 - 3*x[0] + 45

def grad_func(x):
    return 2*x[0]-3

x_0 = 10
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
Количество оценок: 3
Решение: f([1.5]) = 42.75000


задание 4,7

In [26]:

from scipy.optimize import minimize

def func(x):
    return x[0]**4 + 6* x[1]**2 + 10

def grad_func(x):
    return np.array([4 * x[0]**3, 12* x[1]])

x_0 = (100,100)
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))


result = minimize(func, x_0, method='BFGS', jac=grad_func)
# получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Количество оценок: 40
Решение: f([-9.52718297e-03 -2.32170510e-06]) = 10.00000
Статус оптимизации Optimization terminated successfully.
Количество оценок: 37
Решение: f([1.31617159e-02 6.65344582e-14]) = 10.00000


У каждой библиотеки есть свои особенности использования, и большинство задач можно решить с помощью любой из них. Давайте посмотрим все варианты, чтобы у вас всегда был выбор — решим по одной задаче для каждой библиотеки.

In [27]:
values = [4, 2, 1, 7, 3, 6] #стоимости товаров
weights = [5, 9, 8, 2, 6, 5] #вес товаров
C = 15 #вместимость сумки
n = 6 #количество товаров

In [70]:
c = - np.array(values) #изменяем знак, чтобы перейти от задачи максимизации к задаче минимизации
print(c)
A = np.array(weights)  #конвертируем список с весами в массив
print(A)
A = np.expand_dims(A, 0) #преобразуем размерность массива
print(A)
b = np.array([C]) #конвертируем вместимость в массив
print(b)
print(A.shape)
print(b.shape)

[-4 -2 -1 -7 -3 -6]
[5 9 8 2 6 5]
[[5 9 8 2 6 5]]
[15]
(1, 6)
(1,)


In [71]:
from scipy.optimize import linprog
linprog(c=c, A_ub=A, b_ub=b)

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -52.5
              x: [ 0.000e+00  0.000e+00  0.000e+00  7.500e+00  0.000e+00
                   0.000e+00]
            nit: 0
          lower:  residual: [ 0.000e+00  0.000e+00  0.000e+00  7.500e+00
                              0.000e+00  0.000e+00]
                 marginals: [ 1.350e+01  2.950e+01  2.700e+01  0.000e+00
                              1.800e+01  1.150e+01]
          upper:  residual: [       inf        inf        inf        inf
                                    inf        inf]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00]
                 marginals: [-3.500e+00]
 mip_node_count: 0
 mip_dual_bound: 0.0
        mip_gap: 0.0

С помощью CVXPY создадим переменную-массив. Укажем его размерность, а также условие, что все числа в массиве должны быть целыми:

In [72]:
import cvxpy
x = cvxpy.Variable(shape=n, integer = True)

In [73]:
A = A.flatten() # Преобразуем размерность массива
constraint = cvxpy.sum(cvxpy.multiply(A, x)) <= C
total_value = cvxpy.sum(cvxpy.multiply(x, c))

In [74]:
import cvxpy as cp
import scipy

print("Доступные решатели в CVXPY:", cp.installed_solvers())
print("Версия SciPy:", scipy.__version__)

(CVXPY) Jan 24 03:02:03 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Jan 24 03:02:03 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


Доступные решатели в CVXPY: ['CLARABEL', 'OSQP', 'SCIPY', 'SCS']
Версия SciPy: 1.15.3


In [77]:
# После создания задачи problem
try:
    problem.solve(solver=cp.CLARABEL, verbose=True)
    print("Модель решается с непрерывными переменными.")
except Exception as e:
    print(f"Ошибка в постановке задачи (возможно, невыпуклая): {e}")

(CVXPY) Jan 24 03:08:44 PM: Your problem has 6 variables, 1 constraints, and 0 parameters.
(CVXPY) Jan 24 03:08:44 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jan 24 03:08:44 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jan 24 03:08:44 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jan 24 03:08:44 PM: Your problem is compiled with the CPP canonicalization backend.


                                     CVXPY                                     
                                     v1.7.5                                    
Ошибка в постановке задачи (возможно, невыпуклая): Problem is mixed-integer, but candidate QP/Conic solvers ([]) are not MIP-capable.


In [79]:
problem = cvxpy.Problem(cvxpy.Minimize(total_value), constraints=[constraint])
problem.solve()

SolverError: Solver 'SCIPY' failed. Try another solver, or solve with verbose=True for more information.

In [80]:
import cvxpy as cp

# ... (ваш код создания переменных, цели и ограничений) ...

problem = cp.Problem(cp.Minimize(total_value), constraints=[constraint])

# 1. Попробуем CBC (главный кандидат для MIP)
try:
    print("Пробуем решить с помощью CBC...")
    problem.solve(solver='CBC', verbose=True)
except Exception as e:
    print(f"CBC не сработал: {e}")

# 2. Если CBC не установился, попробуем другие варианты
if problem.status is None:
    print("\nПробуем другие доступные решатели...")
    available = cp.installed_solvers()
    print(f"Установленные решатели: {available}")
    
    for solver_name in available:
        try:
            print(f"\nПробуем: {solver_name}")
            problem.solve(solver=solver_name, verbose=False)
            if problem.status in ["optimal", "optimal_inaccurate"]:
                print(f"Успех с {solver_name}!")
                break
        except Exception as e:
            print(f"  {solver_name} ошибка: {e}")
            continue

# 3. Вывод результатов
print("\n" + "="*50)
print("ИТОГИ:")
print("="*50)
print("Статус:", problem.status)
print("Доступные решатели:", cp.installed_solvers())

if problem.status not in ["infeasible", "unbounded", None]:
    print("Целевая функция:", problem.value)
    # Вывод значений переменных, если они есть
    # for var in problem.variables():
    #     print(f"{var.name()}: {var.value}")
elif problem.status is None:
    print("Задача не была решена ни одним решателем.")
else:
    print("Задача неразрешима или неограничена.")

(CVXPY) Jan 24 03:10:00 PM: Your problem has 6 variables, 1 constraints, and 0 parameters.
(CVXPY) Jan 24 03:10:00 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jan 24 03:10:00 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jan 24 03:10:00 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jan 24 03:10:00 PM: Your problem is compiled with the CPP canonicalization backend.


Пробуем решить с помощью CBC...
                                     CVXPY                                     
                                     v1.7.5                                    
CBC не сработал: The solver CBC is not installed.

Пробуем другие доступные решатели...


(CVXPY) Jan 24 03:10:01 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Jan 24 03:10:01 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Jan 24 03:10:01 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Jan 24 03:10:01 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


Установленные решатели: ['CLARABEL', 'OSQP', 'SCIP', 'SCIPY', 'SCS']

Пробуем: CLARABEL
  CLARABEL ошибка: Problem is mixed-integer, but candidate QP/Conic solvers ([]) are not MIP-capable.

Пробуем: OSQP
  OSQP ошибка: Problem is mixed-integer, but candidate QP/Conic solvers ([]) are not MIP-capable.

Пробуем: SCIP
  SCIP ошибка: The solver SCIP is not installed.

Пробуем: SCIPY
  SCIPY ошибка: Solver 'SCIPY' failed. Try another solver, or solve with verbose=True for more information.

Пробуем: SCS
  SCS ошибка: Problem is mixed-integer, but candidate QP/Conic solvers ([]) are not MIP-capable.

ИТОГИ:
Статус: None
Доступные решатели: ['CLARABEL', 'OSQP', 'SCIP', 'SCIPY', 'SCS']
Задача не была решена ни одним решателем.


In [81]:
x = cvxpy.Variable(shape=n, integer=True)
constraint = cvxpy.sum(cvxpy.multiply(A, x)) <= C
x_positive = x >= 0
total_value = cvxpy.sum(cvxpy.multiply(x, c))

problem = cvxpy.Problem(
    cvxpy.Minimize(total_value), constraints=[constraint, x_positive]
)

print(problem.solve())
print(x.value)

-49.0
[-0. -0. -0.  7. -0.  0.]


In [82]:
x = cvxpy.Variable(shape=n, boolean=True)
constraint = cvxpy.sum(cvxpy.multiply(A, x)) <= C
x_positive = x >= 0
total_value = cvxpy.sum(cvxpy.multiply(x, c))

problem = cvxpy.Problem(
    cvxpy.Minimize(total_value), constraints=[constraint, x_positive]
)

print(problem.solve())
print(x.value)

-17.0
[1. 0. 0. 1. 0. 1.]


PuLP

In [64]:
from pulp import LpProblem, LpMaximize, LpVariable, LpInteger
import pulp as pl

In [83]:
problem = LpProblem('Производство машин', pl.LpMaximize)
# 2. Определение переменных (целые числа, неотрицательные)
A = pl.LpVariable('Автомобиль A', lowBound=0, cat=pl.LpInteger)
B = pl.LpVariable('Автомобиль B', lowBound=0, cat=pl.LpInteger)
#Целевая функция
problem += 20000*A + 45000*B 
#Ограничения
problem += 4*A + 5*B <= 30 
problem += 3*A + 6*B <=30
problem += 2*A + 7*B <=30
problem.solve()
print("Количество автомобилей модели А: ", A.varValue)
print("Количество автомобилей модели В: ", B.varValue)
print("Суммарный доход: ", problem.objective.value())  # Правильный способ вывода значения цели
#Количество автомобилей модели А:  1.0
#Количество автомобилей модели В:  4.0
#Суммарный доход:  200000.0

Количество автомобилей модели А:  1.0
Количество автомобилей модели В:  4.0
Суммарный доход:  200000.0
